# Бинарная релевантность

$$\text{precision@k}(q) = \frac{1}{k} \sum_{i=1}^{k} [y_{(i)} = 1]$$

Здесь квадратные скобки $[y_{(i)} = 1]$ обозначают нотацию Айверсона (или функцию-индикатор): выражение равно 1, если условие внутри истинно (документ на позиции i релевантен), и 0, если ложно.



$$\text{AP@k}(q) = \frac{1}{\sum_{i=1}^{k} y_{(i)}} \sum_{i=1}^{k} y_{(i)} \text{precision@i}(q)$$



AP@N это то же самое что AUC-PR.

Да, математически AP (Average Precision) и AUC-PR (Area Under the Precision-Recall Curve) — это практически одно и то же, но между ними есть тонкая техническая разница в способе вычисления.

Поэтому в библиотеках вроде scikit-learn функция average_precision_score используется как более точная и консервативная альтернатива вычислению площади через auc для PR-кривой. В статьях и индустрии, когда говорят про ранжирование и поиск (где мы оперируем дискретными позициями в топе — @N), всегда используют термин AP@N, а не AUC-PR.

# **BM25** 

Документ и запрос можно сравнить, например, путём подсчёта косинусного расстояния между их TF-IDF-представлениями. Более общим способов вычисления близости является функция Okapi BM25. Пусть запрос $q$ состоит из слов $q_1, \dots, q_n$. Тогда его сходство с документов вычисляется как

$$\text{BM25}(q, d) = \sum_{i=1}^{n} \text{IDF}(q_i) \frac{\text{tf}(q_i, d)(k_1 + 1)}{\text{tf}(q_i, d) + k_1 \left(1 - b + b \frac{|D|}{\bar{n}_d}\right)}$$

где $\text{tf}(q_i, d)$ — число вхождений слова $q_i$ в документ $d$, $|D|$ — число документов в выборке, $\bar{n}_d$ — средняя длина документа, а $\text{IDF}$ (inverse document frequency) может вычисляться по формуле

$ \text{IDF}(q_i) = \log \frac{|D|}{|\{d \in D \mid q_i \in d\}|}, $

т.е. как доля документов, содержащих данное слово. Величины $b$ и $k_1$ являются параметрами.

### 1. Базовый вес слова: $IDF(q_i)$
Эта часть отвечает за «ценность» слова. Обратная частота документа (IDF) означает, что редкие слова весят больше. Если пользователь ищет «где купить осциллограф», слово «осциллограф» встретится в базе редко, поэтому совпадение по нему даст огромный балл. Слово «купить» встречается почти везде, его вес будет минимальным. 

### 2. Борьба с переспамом: $ \frac{\text{tf}(q_i, d) \cdot (k_1 + 1)}{\text{tf}(q_i, d) + k_1} $ (упрощенно)
В старых алгоритмах (TF-IDF) зависимость была линейной: если слово «осциллограф» встречается 1 раз — балл 10, если 100 раз — балл 1000. Это приводило к тому, что в топе были SEO-тексты, где ключевое слово написано через каждую строчку.

BM25 ввел **насыщение частоты (TF saturation)** с помощью коэффициента $k_1$. 
Смысл дроби в том, что рост балла замедляется. Первое появление слова дает максимум пользы. Второе — чуть меньше. Пятое — еще меньше. А после 10-го повторения формула выходит на плато (асимптоту). Система говорит: «Я уже поняла, что этот текст про осциллограф, хватит повторять, балл больше не вырастет». Коэффициент $k_1$ (обычно от 1.2 до 2.0) как раз определяет, как быстро наступит это плато. 

### 3. Справедливость к длине: $ 1 - b + b \cdot \frac{|D|}{\bar{n}_d} $
Эта часть находится в знаменателе и называется **нормализацией длины**.
Представьте статью на 100 слов, где слово «Apple» встречается 3 раза, и книгу на 10 000 слов, где «Apple» тоже встречается 3 раза. Очевидно, что короткая статья полностью посвящена Apple, а в книге это слово упомянули вскользь. 

Чтобы уравнять шансы, BM25 делит длину текущего документа ($|D|$) на среднюю длину всех документов в базе ($\bar{n}_d$). 
- Если текст длиннее среднего, знаменатель растет, и итоговый балл за слово **штрафуется** (уменьшается).
- Если текст короче среднего, балл **увеличивается**.

Коэффициент $b$ (от 0 до 1, обычно 0.75) регулирует силу этого штрафа. Если поставить $b=0$, алгоритм вообще перестанет смотреть на длину текста (это называется BM15). Если $b=1$, штраф за длину будет максимальным (BM11).

**Резюме:** Формула берет редкость слова (IDF) и умножает на его количество в тексте (TF), но при этом «обрезает» пользу от лишних повторений ($k_1$) и делает скидку на то, насколько длинный сам текст ($b$).

# 4.2 Попарные методы

### RankNet

Вспомним, что изначально мы определяли задачу ранжирования через пары объектов. Если записывать это формально, то получим функционал ошибки

$ \sum\limits_{(i,j) \in R} [a(x_j) - a(x_i) < 0], $

где $R$ — множество пар, для которых известен порядок. Этот функционал не является дифференцируемым, но мы уже решали такую проблему в линейной классификации — надо заменить индикатор ошибки $[z < 0]$ на его гладкую верхнюю оценку $L(z)$:

$ \sum\limits_{(i,j) \in R} [a(x_j) - a(x_i) < 0] \leqslant \sum_{(i,j) \in R} L(a(x_j) - a(x_i)). $

В качестве оценки $L(z)$ можно брать, например, логистическую функцию $L(x) = \log(1 + e^{-\sigma z})$ с параметром $\sigma > 0$ — в этом случае получим метод RankNet. Оптимизировать данный функционал можно обычным стохастическим градиентным спуском. Если использовать линейную модель $a(x) = \langle w, x \rangle$, то один шаг будет иметь вид

$ w := w + \eta \frac{\sigma}{1 + \exp(\sigma \langle x_j - x_i, w \rangle)} (x_j - x_i). $ 

### В чем суть формулы RankNet?
Формула выглядит так: $ L(x) = \log(1 + e^{-\sigma z}) $, где $z = a(x_i) - a(x_j)$. 

Представьте, что у нас есть два документа по запросу «как варить пельмени»:
- $x_i$ — отличный рецепт от шеф-повара (мы знаем от асессоров, что он должен быть выше).
- $x_j$ — статья про историю пельменей (мы знаем, что она должна быть ниже).

Модель выдает каждому документу какую-то свою внутреннюю оценку (скор). Допустим, модель дала хорошему документу $a(x_i) = 10$, а плохому $a(x_j) = 5$.
Тогда их разница $z = 10 - 5 = 5$. 
Если мы подставим $z = 5$ в формулу RankNet (игнорируя для простоты $\sigma$), мы получим: 
$\log(1 + e^{-5}) \approx \log(1 + 0.006) \approx 0.006$.
**Штраф (ошибка) очень маленький.** Модель молодец, она поставила хороший документ намного выше плохого. 

А теперь представьте, что модель ошиблась и дала плохому документу 10 баллов, а хорошему 5.
Тогда разница $z = 5 - 10 = -5$.
Подставляем в формулу:
$\log(1 + e^{-(-5)}) = \log(1 + e^5) = \log(1 + 148.4) \approx 5$.
**Штраф огромный.** Нейросеть получает сильный «удар по рукам» и при следующем проходе обучения сильно изменит свои внутренние веса. 


### Главная фишка: «уверенность» модели
Почему используется именно такая функция с экспонентой и логарифмом, а не просто линейный штраф?

1. **Если модель чуть-чуть ошиблась** (дала плохому документу 5.1 балла, а хорошему 5.0, то есть $z = -0.1$), штраф будет умеренным.
2. **Если модель жестоко ошиблась и при этом «уверена» в своей правоте** (дала плохому документу 100 баллов, а хорошему 0, то есть $z = -100$), экспонента $e^{100}$ взорвется до гигантских значений. Штраф будет колоссальным. Функция RankNet наказывает модель тем сильнее, чем больше разрыв в баллах не в ту сторону.
   
4. **Асимптота успеха:** Если модель дала правильный порядок с разницей в 10 баллов ($z = 10$), штраф почти нулевой. И если она увеличит разрыв до 100 баллов ($z = 100$), штраф останется почти нулевым. Функция говорит модели: «Ты уже поставила документы в правильном порядке с хорошим запасом, хватит их раздвигать, иди учись на других парах».
   
Параметр $\sigma$ в формуле просто регулирует крутизну этой кривой (насколько резко возрастает штраф). 

Таким образом, RankNet заставляет модель фокусироваться на тех парах документов, порядок которых она перепутала сильнее всего, и игнорировать те пары, которые она уже отсортировала правильно. 

### Вектор направления $(x_j - x_i)$
- $x_i$ и $x_j$ — это векторы признаков документов. Например, у них могут быть признаки $[кол-во\ слов,\ pagerank,\ совпадения\ в\ title]$.
- Разница $(x_j - x_i)$ показывает, в чем именно плохой документ отличается от хорошего. 
- Это **направление градиента**. Оно подсказывает: «Смотри, у хорошего документа $x_i$ признак 'совпадение в title' больше, чем у плохого. Значит, нам нужно увеличить вес этого признака».

## Проблема прошлого подхода: 
1. Допустим, у нас есть пара документов $(d_3, d_5)$, и мы знаем, что $d_3$ чуть-чуть лучше, чем $d_5$.
   
3. Но при этом **оба эти документа — абсолютно нерелевантный мусор** по отношению к запросу пользователя. Они находятся где-то в самом конце выдачи (например, на 90-й и 95-й позиции).
   
5. Если модель перепутает их местами (поставит $d_5$ выше $d_3$), обычный RankNet выпишет ей такой же штраф, как если бы она перепутала самый лучший документ на 1-м месте со 2-м местом.
   
7. **Вывод автора:** «эта пара не должна сильно влиять на ошибку». Пользователю абсолютно всё равно, в каком порядке отсортирован мусор на 10-й странице поиска. Нам критически важно правильно отсортировать топ-10. 

### Переход к вопросу «Как оптимизировать DCG?»
Из-за этой проблемы исследователи поняли, что штрафовать просто за перепутанные пары — неэффективно. Нужно штрафовать **с учетом позиции документа в выдаче**.

Именно это делает метрика **NDCG** (Normalized Discounted Cumulative Gain), которая:
- Дает много баллов за релевантные документы на первых местах.
- Сильно «штрафует» (дисконтирует) баллы за документы, которые оказались далеко внизу.

В алгоритмам **LambdaRank/LambdaMART** в формулу градиента (которую мы разбирали выше) просто домножают на то, **как сильно изменится метрика NDCG, если мы поменяем эти два документа местами**. 
- Если мы меняем местами мусор на 90-м месте — NDCG почти не изменится (штраф нулевой).
- Если мы меняем 1-е и 2-е место — NDCG рухнет (штраф огромный).

## Запишем LambdaRank

Существует эмпирическое наблюдение, позволяющее перейти к оптимизации произвольной метрики ранжирования $F$. Оказывается, для этого надо домножить смещение на изменение метрики $\Delta F_{ij}$, которое произойдёт при перестановке $x_i$ и $x_j$ местами в ранжировании:

$ w := w + \eta \frac{\sigma}{1 + \exp(\sigma \langle x_j - x_i, w \rangle)} |\Delta F_{ij}| (x_j - x_i). $

Данный метод носит название **LambdaRank**. 

### Как мы рассчитываем компонент $|\Delta F_{ij}|$?

Компонент $|\Delta F_{ij}|$ — это модуль изменения какой-либо метрики качества (обычно NDCG, но может быть и ERR, MAP и т.д.), если мы искусственно поменяем местами в итоговой выдаче два документа $x_i$ и $x_j$ .

Представьте, что текущая модель отсортировала топ-3 так:
1. Документ А (истинная релевантность = 1)
2. Документ Б (истинная релевантность = 4)
3. Документ В (истинная релевантность = 0)

Считаем текущий DCG для первых двух позиций (документы А и Б):
- Позиция 1 (Док А): $\frac{2^1 - 1}{\log_2(1 + 1)} = \frac{1}{1} = 1$
- Позиция 2 (Док Б): $\frac{2^4 - 1}{\log_2(2 + 1)} = \frac{15}{1.58} \approx 9.49$
- **Суммарный DCG сейчас = $1 + 9.49 = 10.49$**.

Теперь мы хотим понять $|\Delta \text{DCG}|$ для пары (Док А, Док Б). Что будет, если мы поменяем их местами (Док Б встанет на 1-е место, а Док А на 2-е)?
Считаем новый DCG:
- Позиция 1 (теперь Док Б): $\frac{2^4 - 1}{\log_2(2)} = \frac{15}{1} = 15$
- Позиция 2 (теперь Док А): $\frac{2^1 - 1}{\log_2(3)} = \frac{1}{1.58} \approx 0.63$
- **Новый DCG = $15 + 0.63 = 15.63$**.

Вычисляем разницу:
$ |\Delta F_{ij}| = |15.63 - 10.49| = 5.14 $

### Что дает это число?
Число 5.14 — это и есть наш $|\Delta F_{ij}|$. Мы берем стандартный градиент RankNet и умножаем его на 5.14. Модель получает огромный пинок: «Смотри, если бы ты поставила Документ Б выше Документа А, качество выдачи взлетело бы на 5.14 балла! Немедленно меняй свои веса $w$ в эту сторону!».

Если бы мы меняли местами документы на 99-й и 100-й позиции, знаменатель в формуле DCG ($\log_2(100)$) был бы огромным. Из-за этого изменение метрики $\Delta \text{DCG}$ составило бы, например, 0.001. Градиент умножился бы на 0.001, и модель бы поняла: «Эта перестановка почти ни на что не влияет, не буду тратить силы на изменение весов ради этого мусора». 

### Как происходит шаг градиентного спуска?

### 1. Формирование батча (эпохи)
Модель не учится на всех триллионах возможных пар одновременно. В качестве одного «шага» (или батча) в ранжировании обычно берется **один поисковый запрос**. [education.yandex](https://education.yandex.ru/handbook/ml/article/zadacha-ranzhirovaniya)
Допустим, на этой итерации мы выбрали из обучающей выборки запрос «рецепт блинов». Для этого запроса у нас есть, например, 100 документов с известными оценками асессоров (от 0 до 4). 

### 2. Генерация пар внутри запроса
Для этого конкретного запроса алгоритм генерирует все возможные пары документов $(x_i, x_j)$, у которых разная релевантность (оценки асессоров). 
- Пары вроде (оценка 4, оценка 1) — берем в работу, мы точно знаем, какой из них лучше.
- Пары вроде (оценка 3, оценка 3) — игнорируем, по ним нельзя понять правильный порядок.

### 3. Оценка моделью и расчет градиентов
Модель делает проход (forward pass) и выдает каждому из 100 документов свой текущий балл $a(x)$.
Затем для *каждой валидной пары* $(i, j)$ из этого запроса вычисляется ее маленькая ошибка и ее маленький градиент (направление, куда надо сдвинуть веса), по формуле, которую мы разбирали. 

### 4. Агрегация («Лямбды») и шаг
Если бы мы делали шаг весов $\Delta w$ после каждой отдельной пары, модель бы мотало из стороны в сторону: сначала пара (1, 2) потянула веса влево, потом пара (1, 3) потянула вправо. Это очень медленно и неэффективно. 

Вместо этого RankNet делает **агрегацию**:
Для каждого конкретного документа $x_i$ суммируются все «тянущие» силы от всех его пар. 
- Если документ хороший (релевантность 4), то в 99 парах с плохими документами формула скажет: «Тяни его оценку вверх!». Вектор силы (обозначаемый как $\lambda$, лямбда) будет большим и направленным вверх.
- Если документ средний (релевантность 2), то плохие документы будут «толкать» его вверх, а отличные (релевантность 4) будут «толкать» его вниз. Итоговый вектор $\lambda$ сложится из этих сил. 

В итоге, вычислив эти суммарные силы для каждого из 100 документов по одному запросу, алгоритм делает **один общий шаг градиентного спуска (обновление весов)**. 
После этого берется следующий запрос («купить айфон»), и процесс повторяется. 

**Краткий ответ:** Шаг градиента делается не по одной паре, а по сумме градиентов всех пар, относящихся к одному поисковому запросу. Это позволяет обновлять модель стабильно и быстро.

Хотите узнать, как в эту схему встраиваются градиентные деревья (бустинг) при переходе к LambdaMART?

## Разберем на примере шаг градиентного спуска

### Шаг 1: Исходные данные батча
Допустим, запрос: *"как сварить яйцо"*.
Для этого запроса в нашем обучающем датасете есть 4 документа, которые асессоры (люди) заранее разметили по шкале релевантности от 0 (мусор) до 3 (идеально):

- **Док А**: Оценка 3 (пошаговый рецепт варки яиц)
- **Док Б**: Оценка 2 (статья про пользу вареных яиц)
- **Док В**: Оценка 1 (рецепт яичницы)
- **Док Г**: Оценка 1 (рецепт омлета)

Каждый из этих документов уже превращен в вектор признаков (TF-IDF, длина текста, наличие картинок и т.д.).

### Шаг 2: Модель делает предсказание
Модель со своими текущими весами смотрит на эти 4 документа и выдает каждому свою математическую оценку (скор). Допустим, она пока работает плохо и выдала такие баллы:

- Док А (оценка 3): **Скор модели 0.5**
- Док Б (оценка 2): **Скор модели 0.8**
- Док В (оценка 1): **Скор модели 0.4**
- Док Г (оценка 1): **Скор модели 0.9** 

Модель страшно ошиблась: она поставила мусорный Док Г на первое место (скор 0.9), а лучший Док А — почти в конец (скор 0.5).

### Шаг 3: Как формируются пары?
Алгоритм начинает перебирать все возможные комбинации этих четырех документов. Главное правило генерации пар в RankNet: **мы создаем пару, только если у документов РАЗНАЯ оценка от асессоров**. [logic.pdmi.ras](https://logic.pdmi.ras.ru/~sergey/teaching/mlspsu23/11-ranking.pdf)

Давайте сгенерируем все валидные пары для нашего батча:
1. **Пара (А, Б):** Оценки (3 > 2). Берем!
2. **Пара (А, В):** Оценки (3 > 1). Берем!
3. **Пара (А, Г):** Оценки (3 > 1). Берем!
4. **Пара (Б, В):** Оценки (2 > 1). Берем!
5. **Пара (Б, Г):** Оценки (2 > 1). Берем!
6. **Пара (В, Г):** Оценки (1 == 1). **Игнорируем!** Мы не знаем, кто из них лучше, поэтому штрафовать модель за их перестановку нельзя.

Итого, для этого батча (одного запроса) мы сгенерировали **5 пар**. [naimee](https://naimee.ai/hrzavtra/ii-rezyume-obuchenie-modeli-ranzhirovat-kandidatov)

### Шаг 4: Расчет градиентов (ошибок) для пар
Теперь алгоритм смотрит на каждую из 5 пар и вычисляет градиент по формуле, которую мы разбирали:

- **Пара (А, Б):** Док А должен быть выше Дока Б (3 > 2). Но модель дала А скор 0.5, а Б скор 0.8. Модель ошиблась. Формула генерирует сильный штраф и градиент: «Тяни веса так, чтобы скор А вырос, а скор Б упал».
- **Пара (А, В):** Док А должен быть выше Дока В (3 > 1). Модель дала А скор 0.5, а В скор 0.4. Модель угадала порядок, но разница микроскопическая (0.1). Формула выдаст слабый градиент: «Молодец, но раздвинь их чуть сильнее».
- И так для всех 5 пар.

### Шаг 5: Агрегация в "Лямбды" и обновление модели
Вместо того, чтобы дергать модель 5 раз из-за каждой пары, алгоритм суммирует все векторы градиентов:
- Док А участвует в трех парах, и во всех трех парах он "победитель" по мнению асессоров, но проигравший по скорам модели. Суммарный вектор (лямбда) для Дока А будет с огромной силой тянуть его веса вверх.
- Док Г участвует в двух парах, и в обеих он "проигравший" по мнению асессоров, но модель дала ему огромный скор (0.9). Суммарный вектор обрушит веса Дока Г вниз.

После того как эти суммарные "силы" для каждого из 4 документов посчитаны, делается **один шаг градиентного спуска (обновление весов модели)**. 
Затем берется следующий батч (например, запрос "купить машину" и 200 документов к нему), и всё повторяется.

Да, именно так! Значение $|\Delta F_{ij}|$ (например, изменение NDCG) **пересчитывается заново для каждой конкретной пары документов** внутри батча на каждой итерации обучения  [habr](https://habr.com/ru/companies/postgrespro/articles/857998/).

Это звучит как колоссальный объем вычислений, но на самом деле математика этого процесса очень элегантно оптимизирована. Давайте разберем, как это происходит на практике (например, в библиотеках вроде LightGBM или CatBoost, которые реализуют LambdaMART).

### Как это происходит технически (шаг за шагом)

**1. Сортировка текущего батча**
Когда модель на текущем шаге выдала всем документам по запросу свои баллы (скоры), алгоритм **один раз сортирует** все документы этого запроса по убыванию предсказанного скора. [habr](https://habr.com/ru/companies/postgrespro/articles/857998/)
Таким образом мы получаем текущую *позицию (ранг)* каждого документа в выдаче.

**2. Предрасчет идеального IDCG**
Поскольку истинные оценки асессоров известны заранее, алгоритм может один раз (еще до начала обучения) посчитать идеальный DCG для этого запроса — то есть **IDCG** (когда все документы отсортированы идеально). Он нам нужен, чтобы делить на него обычный DCG и получать нормализованный NDCG. [edu.51cto](https://edu.51cto.com/article/note/3909.html)

**3. Быстрый пересчет $\Delta \text{NDCG}$ для пары**
Теперь алгоритм берет пару $(i, j)$. Допустим, документ $i$ сейчас стоит на позиции 2, а документ $j$ — на позиции 5. Мы знаем их истинные релевантности: пусть у $i$ релевантность 3, а у $j$ релевантность 1.

Чтобы узнать, как изменится общий NDCG всего запроса, если мы поменяем их местами, **нам не нужно пересчитывать метрику для всех 100 документов**. Документы на позициях 1, 3, 4 и с 6 по 100 останутся на своих местах, их вклад в формулу не изменится! [edu.51cto](https://edu.51cto.com/article/note/3909.html)

Изменится только вклад позиций 2 и 5. Математически $\Delta \text{DCG}$ для этой пары вычисляется моментально:
$ |\Delta \text{DCG}| = \left| \left( \frac{2^{rel_i} - 1}{\log_2(pos_j + 1)} + \frac{2^{rel_j} - 1}{\log_2(pos_i + 1)} \right) - \left( \frac{2^{rel_i} - 1}{\log_2(pos_i + 1)} + \frac{2^{rel_j} - 1}{\log_2(pos_j + 1)} \right) \right| $
Делим этот результат на предрассчитанный IDCG и получаем наш $|\Delta \text{NDCG}_{ij}|$  [edu.51cto](https://edu.51cto.com/article/note/3909.html). Эта операция занимает микросекунды.

### Зачем это делается для каждой пары?
Потому что ценность перестановки кардинально зависит от текущего положения документов. [edu.51cto](https://edu.51cto.com/article/note/3909.html)
- Если модель поставила крутой документ на 2-е место, а мусорный на 1-е, то перестановка пары (1, 2) даст **огромный $\Delta \text{NDCG}$**. Градиент (лямбда) для этой пары будет гигантским. [edu.51cto](https://edu.51cto.com/article/note/3909.html)
- Если в том же запросе есть пара других документов, которые модель поставила на 98-е и 99-е места, их перестановка даст **микроскопический $\Delta \text{NDCG}$**. Их лямбда почти обнулится. [ffineis.github](https://ffineis.github.io/blog/2021/05/01/lambdarank-lightgbm.html)

### Итог (Агрегация)
Посчитав этот градиент (домноженный на $\Delta \text{NDCG}$) для каждой валидной пары, алгоритм суммирует эти векторы для каждого конкретного документа, получая итоговый градиент $\lambda_i$ (насколько сильно и в какую сторону нужно толкать оценку конкретного документа). А затем по этим градиентам делается шаг градиентного спуска (или строится новое дерево решений, если мы говорим о LambdaMART). [ffineis.github](https://ffineis.github.io/blog/2021/05/01/lambdarank-lightgbm.html)

Получается, что "эмпирическое наблюдение" авторов LambdaRank сэкономило вычислительные ресурсы: вместо того чтобы пытаться оптимизировать недифференцируемую метрику NDCG напрямую (что математически невозможно), они просто взяли гладкие градиенты RankNet и взвесили их на $\Delta \text{NDCG}$, получив идеальный механизм обучения. [logic.pdmi.ras](https://logic.pdmi.ras.ru/~sergey/teaching/mlspsu22/17-ranking.pdf)

# MART


### 2. Исходные данные (Датасет)
Слева написана формула:
$ D = \{ (\bar{x}_n, y_n) \}_{n=1}^{N} $  

Это обучающая выборка (датасет). 
- $N$ — количество примеров (документов).
- $\bar{x}_n$ — вектор признаков $n$-го документа.
- $y_n$ — целевая переменная (то, что мы хотим предсказать). В случае обычного машинного обучения это может быть цена квартиры. В случае нашего поиска (LambdaMART) в роли $y_n$ будут выступать те самые **лямбды ($\lambda_i$)**, которые мы вычисляли на предыдущих шагах.

### 3. Как строится дерево (разбиение узла)
В центре нарисован узел дерева $D$, который делится на две ветки (левую и правую). 
Чтобы разбить данные, алгоритм перебирает все возможные признаки (например, $j$ — номер признака от 1 до $d$) и все возможные пороги разбиения $t$. 

- **Левая ветка:** туда уходят все документы, у которых значение признака $x_{nj} \le t$. 
- **Правая ветка:** туда уходят все документы, у которых $x_{nj} > t$. 

### 4. Что будет в листьях?
Под ветками написаны значения:
$ \mu_L = \text{Avg}(y_n), \quad \mu_R $  

Это ответы, которые выдаст лист. В классическом MART ответ в левом листе ($\mu_L$) — это просто **среднее арифметическое** (Average) всех целевых значений $y_n$ (наших лямбд), которые попали в этот лист. Аналогично для правого листа ($\mu_R$). 

### 5. Как выбирается лучшее разбиение (Функция потерь $L_j$)
Внизу написана формула:

$$ L_j = \sum_{n \in L} (y_n - \mu_L)^2 + \sum_{n \in R} (y_n - \mu_R)^2 \longrightarrow \min_{t, j} $$

Это критерий качества разбиения (MSE — среднеквадратичная ошибка). 
Алгоритм смотрит на левый лист ($L$) и считает сумму квадратов разностей между истинными $y_n$ и предсказанием листа $\mu_L$. То же самое делает для правого листа ($R$).
Затем алгоритм перебирает все признаки $j$ и все пороги $t$, пытаясь найти такое разбиение, при котором **ошибка $L_j$ будет минимальной**. 


### Слайд 1. Введение: MART 
**MART (Multiple Additive Regression Trees)** — это градиентный бустинг, сделанный на регрессионных деревьях.
Здесь показана главная формула бустинга:
$$ F_M(\mathbf{x}) = \sum_{m=1}^{M} \alpha_m f_m(\mathbf{x}) $$
- $ \mathbf{x} $ — вектор признаков нашего объекта (документа).
- $ F_M(\mathbf{x}) $ — итоговое предсказание ансамбля из $ M $ деревьев.
- $ f_m(\mathbf{x}) $ — предсказание одного конкретного (m-го) регрессионного дерева.
- $ \alpha_m $ — вес этого дерева в ансамбле (насколько сильно мы ему доверяем).
Суть бустинга в том, что мы строим деревья по очереди, и каждое новое дерево ($ f_m $) учится исправлять ошибки, которые допустили все предыдущие деревья вместе взятые. 

### Слайд 2. Шаг бустинга: как обучать новое дерево 
У нас уже есть $ m $ обученных деревьев, их сумма дает предсказание $ F_m $. Нам нужно обучить новое дерево $ F_{m+1} $ (здесь на слайде небольшая путаница в обозначениях: $ \Delta F $ — это и есть наше новое дерево $ f_{m+1} $, которое мы хотим добавить). 
Мы хотим минимизировать функцию ошибки (потерь) $ C $.
Идея: если мы рассматриваем текущее предсказание $ F_m $ как «точку», то мы хотим сдвинуться от нее в сторону уменьшения ошибки. Куда двигаться? В направлении **антиградиента** (отрицательной производной) функции потерь. 
Поэтому новое дерево должно предсказывать значения:
$$ \Delta F = -\eta \frac{\partial C(F_m)}{\partial F_m} $$
То есть новое дерево учится предсказывать **производную функции ошибки** (так называемые псевдо-остатки или сдвиги).

### Слайд 3. Пример: бинарная классификация 
Здесь автор переходит от абстрактной функции потерь к конкретной задаче.
- У нас есть классы $ y_i \in \{+1, -1\} $.
- Модель выдает вероятности $ p_+ $ (вероятность класса +1) и $ p_- $ (вероятность класса -1).
- Функция потерь — **перекрёстная энтропия (Log Loss)**:
$$ L(y, F) = -I_+ \log p_+ - I_- \log p_- $$
Где $ I_+ $ и $ I_- $ — индикаторы истинного класса (равны 1, если класс истинный, и 0 иначе). 

### Слайд 4. Логистическая регрессия 
Вместо того чтобы дерево предсказывало вероятности напрямую (что неудобно, так как они от 0 до 1), оно предсказывает так называемые логарифмы шансов (log odds) — функцию $ F_m(\mathbf{x}) $. 
Вероятности связаны с предсказанием дерева через сигмоиду:
$$ p_+ = \frac{1}{1 + e^{-2\alpha F_m(\mathbf{x})}} $$
Подставляя это в формулу перекрестной энтропии из Слайда 3, мы получаем элегантную формулу функции потерь:
$$ L(y, F) = \log(1 + e^{-2 y \alpha F_m(\mathbf{x})}) $$
Это очень похоже на ту логистическую функцию, которую мы видели в RankNet! Чем больше $ y $ и $ F $ не совпадают по знаку (модель ошиблась), тем больше значение экспоненты и тем больше штраф. 

### Слайд 5. Берем производную (считаем псевдо-остатки) 
Чтобы применить идею со Слайда 2, нам нужно взять производную этой функции потерь по предсказанию $ F(\mathbf{x}) $.
$$ \bar{y}_i = \frac{\partial L}{\partial F} = \frac{2 y_i \alpha}{1 + e^{2 y_i \alpha F_m(\mathbf{x})}} $$
Эти $ \bar{y}_i $ — это наши **псевдо-остатки** (в терминах LambdaMART это и есть те самые **лямбды $\lambda_i$**). 
Для каждого объекта обучающей выборки мы считаем это число. И затем мы **строим новое регрессионное дерево, которое пытается предсказать (смоделировать) эти числа $\bar{y}_i$** по признакам $ \mathbf{x} $. 

### Слайд 6. Выбор оптимального шага в листьях
Итак, дерево построено. Оно разбило объекты по листьям. Теперь нам нужно выбрать оптимальный «ответ» $ \gamma $ (сдвиг) для каждого конкретного листа, чтобы минимизировать суммарную потерю всех объектов, попавших в этот лист.
$$ \gamma_{lm} = \arg\min_{\gamma} \sum \log(1 + e^{2 y_i \alpha (F_m + \gamma)}) $$
Поскольку эта функция не решается аналитически (ее нельзя просто приравнять к нулю и найти $ \gamma $), мы используем метод приближения — **метод Ньютона-Рапсона**. Он позволяет найти минимум функции с помощью первой и второй производных. 

### Слайд 7. Итоговая формула для листа (Ньютон-Рапсон) 
Делая один шаг метода Ньютона ($ \gamma = -\frac{g'}{g''} $), мы получаем итоговое значение, которое должно быть записано в каждом листе дерева:
$$ \gamma_{lm} = \frac{\sum \bar{y}_i}{\sum |\bar{y}_i| (2\alpha - |\bar{y}_i|)} $$
- Числитель — это просто сумма наших градиентов (псевдо-остатков $\bar{y}_i$) в этом листе.
- Знаменатель — это сумма вторых производных (гессианов) в этом листе. 

### Резюме всей цепочки:
1. Берем текущие предсказания ансамбля.
2. Считаем ошибку (градиенты $\bar{y}_i$) для каждого объекта (Слайд 5).
3. Строим новое дерево, которое группирует объекты с похожими признаками и градиентами в одни листья.
4. Вычисляем оптимальный ответ для каждого листа по формуле Ньютона (Слайд 7).
5. Прибавляем это дерево к нашему ансамблю и повторяем процесс. 

В LambdaMART происходит абсолютно то же самое, только вместо обычной функции потерь (Слайд 4) используется сдвинутая на $\Delta \text{NDCG}$ функция из RankNet!

## DSSM

$$
p(d \mid q) = \frac{\exp(\beta \,\alpha(q, d))}{\sum_{d' \in D} \exp(\beta \,\alpha(q, d'))}
$$

- $\alpha(q,d)$ — скалярная **сходство** между запросом и документом (ниже ты рисуешь косинус).
- $\beta > 0$ — «температура»: чем больше, тем более «жёстким» становится softmax (разница в $\alpha$ сильнее влияет на вероятности).


$$
-\log \prod_{(q,d) \in R} p(d \mid q) \to \min
$$

- $R$ — множество обучающих пар «правильный документ для запроса».
- Это и есть кросс‑энтропийный loss: мы максимизируем вероятность правильного $d$ на фоне кучи кандидатов.

Ниже — модификация знаменателя: вместо суммы по **всем** документам берём «приближённую» сумму по одному позитиву + пачке негативов (negative sampling). Так как «слишком много слагаемых», поэтому:

$$
p(d \mid q) \approx \frac{\exp(\beta \,\alpha(q, d))}
{\exp(\beta \,\alpha(q, d)) + \sum_{(q, d')\ \text{не было клика}} \exp(\beta \,\alpha(q, d'))}
$$

То есть softmax только по позитиву и выбранным негативам.

## Сама двухбашенная модель


- Слева $q$ проходит через «много слоёв» и превращается в вектор $v_q$.
- Справа $d$ проходит через свои «много слоёв» и превращается в $v_d$.
- Сверху пометка shared weights — это как раз дискуссия «шарим ли веса между башнями», но в общем случае башни разные.
- В центре считаешь косинус:

$$
\alpha(q,d) = \frac{\langle v_q, v_d \rangle}{\|v_q\|\ \|v_d\|}
$$

Это и есть тот самый скор $\alpha(q,d)$, который идёт в softmax.

Интуитивно: модель учит такие преобразования $q \mapsto v_q$ и $d \mapsto v_d$, чтобы **правильные** пары имели высокий косинус, а все остальные — низкий, причём это формализовано через вероятностную модель $p(d \mid q)$ и cross‑entropy.




### Почему нужен negative sampling

Формула выше суммирует по **всем** документам $D$. В реальных задачах $D$ — миллионы/миллиарды объектов, и:

- невозможно на каждом шаге читать и прогонять через модель все документы;
- даже если эмбеддинги документов уже предвычислены, суммировать миллионы экспонент на каждый запрос дорого.

Поэтому вместо «softmax по всему каталогу» делают softmax по **одному позитиву + набору негативов** (обычно десятки–сотни) из каталога:

- либо случайные негативы;
- либо «hard negatives» — кандидаты, которые модель уже считает похожими, но клика/релевантности не было.

Это и есть negative sampling: мы приближённо оптимизируем тот же критерий, но считаем нормировочную константу не по всему $D$, а по небольшой подвыборке. [aclanthology](https://aclanthology.org/2023.emnlp-industry.72.pdf)

Смысл:

- сделать обучение вычислительно подъёмным;
- фокусировать градиент на отличении позитива от типичных/сложных негативов, а не от всего каталога сразу;
- приблизить «идеальный» softmax‑loss так, чтобы он работал на больших данных.

## Triplet Loss

Если вместо softmax‑loss ты используешь triplet loss, меняется и постановка, и смысл обучения эмбеддингов.

## Формула и интуиция triplet loss

Берём три эмбеддинга:

- anchor $a = v_q$ (обычно запрос или юзер),
- positive $p = v_{d^+}$ — релевантный документ/айтем,
- negative $n = v_{d^-}$ — нерелевантный.

Вводим расстояния $d(a,p)$ и $d(a,n)$ (например, $1-\cos$ или L2) и лосс:

$$
L_{\text{triplet}} = \max\bigl(0,\ d(a,p) - d(a,n) + m\bigr),
$$

где $m > 0$ — margin.  

Смысл:

- если $d(a,n) \ge d(a,p) + m$, лосс ноль, триплет уже «хорошо ранжирован»;
- если негатив слишком близко или позитив слишком далеко, градиент двигает эмбеддинги, чтобы сделать $a$ ближе к $p$ и дальше от $n$ минимум на margin.

То есть мы **не пытаемся оценить абсолютную вероятность** $p(d \mid q)$; нас волнует только относительное неравенство «позитив ближе негатива». Это pairwise/ranking‑критерий.

## Сравнение с softmax‑loss

Softmax‑loss:

- моделирует $p(d \mid q)$, нормированную по набору кандидатов;
- градиент зависит от всех негативов в нормировке;
- логически соответствует «мультклассовой классификации по документам».

Triplet loss:

- не строит вероятности, только хочет, чтобы все «правильные» объекты были ближе «неправильных»;
- не требует суммировать по всем негативам, достаточно триплетов;
- в чистом виде не говорит, насколько хорошо распределены все документы относительно друг друга, только про удовлетворение неравенств для выбранных триплетов.

В рекомендациях и DSSM‑подобных штуках его часто используют именно потому, что **для ранжирования важен относительный порядок, а не калиброванные вероятности**.

## Связь с negative sampling

В softmax‑подходе ты делаешь negative sampling, чтобы приблизить нормировку по всему каталогу подмножеством негативов. В triplet‑lossе:

- sampling «зашит» в саму конструкцию: каждый триплет — это выбор одного позитива и одного негатива;
- огромная часть качества определяется тем, *как* ты выбираешь негативы (random / hard / semi‑hard mining, внутри батча и т.п.);
- по сути это тот же принцип: мы не можем учитывать все негативы, поэтому учимся на небольшом, но «грамотно выбранном» их подмножестве.

Так что, если перейти на triplet loss, ты избавляешься от явной нормировки softmax, но задача выбора информативных негативов (negative / hard mining) никуда не исчезает — она становится центральной.
